# Demographic matching: iEEG to MEG

Select one distinct MEG participant for each eligible iEEG participant. With only three covariates, direct optimal matching is a transparent alternative to estimating a propensity score. The default is **exact matching on sex and binary musician status**, followed by minimization of total absolute age differences. Unlike greedy nearest-neighbor matching, the assignment minimizes the objective jointly across all pairs.

For assignment indicators $x_{ij}$, solve
$$\min_x \sum_{i=1}^{n_I}\sum_{j=1}^{n_M}x_{ij}|a_i-a_j|,\qquad
\sum_j x_{ij}=1,\quad \sum_i x_{ij}\leq1,\quad x_{ij}\in\{0,1\}.$$
Incompatible sex/musician pairs are forbidden; an optional age caliper also forbids pairs beyond a specified difference. If infeasible, stop and review the constraints rather than silently dropping participants or relaxing matching. This minimizes total age distance, not the maximum distance or every balance statistic.

**Measurement caveat:** instrument playing in iEEG and self-reported musician status in MEG are proxies that may not be equivalent. Verify questionnaire definitions and coding before interpreting musical-expertise balance. Demographic matching does not remove clinical or other unmeasured differences between modalities.

References: [SciPy assignment solver](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.linear_sum_assignment.html); [Stuart and Rubin, matching methods](https://www.biostat.jhsph.edu/~estuart/StuRub_MatchingChapter_07.pdf).


In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from IPython.display import display

# Update these paths for the machine holding the demographic and epoch files.
IEEG_EPOCH_DIR = Path('/users/barbara/Desktop/MIB_iEEG/iEEGvsMEG/ieeg_shortWOBS_fs250')
IEEG_DEM_PATH = Path('/users/barbara/Desktop/MINDLAB2021_MEG-TempSeqAges/scratch/learning_bach_iEEG/misc/clean_dataset.csv')
MEG_DEM_PATH = Path('/users/barbara/Desktop/MIB_iEEG/iEEGvsMEG/MEG/SubjectsDataCollection_APR2020_TEMP.xlsx')
MAX_AGE_GAP = None  # Years; e.g., 5. Choose before examining downstream outcomes.
EXPECTED_N = {'iEEG': 28, 'MEG': 78}  # Warnings identify departures from the intended cohorts.
EXPORT_RESULTS = False
OUTPUT_DIR = Path('out/demographic_matching')


In [ ]:
# Read the original files; preserve raw tables for auditing.
for path in (IEEG_EPOCH_DIR, IEEG_DEM_PATH, MEG_DEM_PATH):
    if not path.exists():
        raise FileNotFoundError(f'Update the input path: {path}')
ieeg_subjects = sorted(p.name.removesuffix('_epochs.p') for p in IEEG_EPOCH_DIR.glob('*_epochs.p'))
if not ieeg_subjects:
    raise ValueError('No iEEG epoch files found; verify IEEG_EPOCH_DIR.')
ieeg_dem = pd.read_csv(IEEG_DEM_PATH)
meg_dem = pd.read_excel(MEG_DEM_PATH, sheet_name='Background questionnaire')
ieeg_dem_ = ieeg_dem[['subject', 'age', 'sex', 'instrument']].copy()
ieeg_dem_['subject'] = ieeg_dem_['subject'].astype('string').str.strip()
ieeg_dem_ = ieeg_dem_[ieeg_dem_['subject'].isin(ieeg_subjects)].copy()
missing_demographic_ids = sorted(set(ieeg_subjects) - set(ieeg_dem_['subject']))
print('Epoch participants absent from demographic table:', missing_demographic_ids)
meg_dem_ = meg_dem[['Subject', 'Age', 'Sex', '12. Are you a musician?']].rename(
    columns={'Subject': 'subject', 'Age': 'age', 'Sex': 'sex',
             '12. Are you a musician?': 'instrument'}).copy()
# Inspect source labels before normalization, especially the iEEG instrument field.
for cohort, frame in [('iEEG', ieeg_dem_), ('MEG', meg_dem_)]:
    print(f'\n{cohort}: {len(frame)} source rows')
    for column in ['sex', 'instrument']:
        display(frame[column].value_counts(dropna=False).rename(f'{cohort}: {column}'))


## Validate and harmonize

Unknown labels cause an error. Missing demographic values are reported and excluded, rather than imputed. The previous notebook treated missing iEEG instrument entries as “No”; this implementation retains them as missing unless `IEEG_MISSING_INSTRUMENT_MEANS_NO` is explicitly enabled after checking the data dictionary. List genuine instrument names in `IEEG_INSTRUMENT_LABELS` after reviewing the source labels above. Numeric category codes are not inferred automatically.


In [ ]:
IEEG_MISSING_INSTRUMENT_MEANS_NO = False
IEEG_INSTRUMENT_LABELS = set()  # Example: {'piano', 'guitar'}; lowercase source labels.
SEX_LABELS = {'f': 'Female', 'female': 'Female', 'woman': 'Female',
              'm': 'Male', 'male': 'Male', 'man': 'Male'}
MUSIC_LABELS = {'yes': 'Yes', 'y': 'Yes', 'true': 'Yes',
                'no': 'No', 'n': 'No', 'false': 'No'}

def normalize_category(series, mapping, label):
    values = series.astype('string').str.strip().str.lower().replace('', pd.NA)
    unknown = sorted(set(values.dropna()) - set(mapping))
    if unknown:
        raise ValueError(f'{label}: unmapped values {unknown}. Verify coding and update the label map.')
    return values.map(mapping)

def prepare_demographics(frame, cohort):
    out = frame[['subject', 'age', 'sex', 'instrument']].copy()
    out['subject'] = out['subject'].astype('string').str.strip().replace('', pd.NA)
    if out['subject'].isna().any() or out['subject'].duplicated().any():
        raise ValueError(f'{cohort}: subject identifiers must be nonmissing and unique.')
    out['age'] = pd.to_numeric(out['age'], errors='raise')
    if ((out['age'].notna()) & (~np.isfinite(out['age']) | (out['age'] <= 0))).any():
        raise ValueError(f'{cohort}: ages must be finite and positive.')
    out['sex'] = normalize_category(out['sex'], SEX_LABELS, f'{cohort} sex')
    music_map = MUSIC_LABELS.copy()
    if cohort == 'iEEG':
        music_map.update({name: 'Yes' for name in IEEG_INSTRUMENT_LABELS})
        if IEEG_MISSING_INSTRUMENT_MEANS_NO:
            out['instrument'] = out['instrument'].fillna('No')
    out['instrument'] = normalize_category(out['instrument'], music_map, f'{cohort} instrument')
    excluded = out[out[['age', 'sex', 'instrument']].isna().any(axis=1)].copy()
    eligible = out.dropna().sort_values('subject').reset_index(drop=True)
    if len(eligible) < 2:
        raise ValueError(f'{cohort}: fewer than two complete participants.')
    if len(eligible) != EXPECTED_N[cohort]:
        warnings.warn(f'{cohort}: {len(eligible)} eligible participants; expected {EXPECTED_N[cohort]}. Review exclusions.')
    return eligible, excluded

ieeg_eligible, ieeg_excluded = prepare_demographics(ieeg_dem_, 'iEEG')
meg_eligible, meg_excluded = prepare_demographics(meg_dem_, 'MEG')
print('Excluded iEEG rows:'); display(ieeg_excluded)
print('Excluded MEG rows:'); display(meg_excluded)


In [ ]:
def optimal_demographic_match(ieeg, meg, max_age_gap=None):
    """Exact sex/musician matching, minimum total age gap, no replacement."""
    if len(ieeg) > len(meg):
        raise ValueError('Fewer eligible MEG than iEEG participants.')
    if max_age_gap is not None and (not np.isfinite(max_age_gap) or max_age_gap < 0):
        raise ValueError('max_age_gap must be None or a finite nonnegative number.')
    counts = pd.concat([
        ieeg.groupby(['sex', 'instrument']).size().rename('iEEG'),
        meg.groupby(['sex', 'instrument']).size().rename('MEG')], axis=1).fillna(0).astype(int)
    if (counts['iEEG'] > counts['MEG']).any():
        raise ValueError('Insufficient MEG participants in exact-match strata:\n' + counts.to_string())
    gaps = np.abs(ieeg['age'].to_numpy(float)[:, None] - meg['age'].to_numpy(float)[None, :])
    allowed = np.ones_like(gaps, dtype=bool)
    for column in ['sex', 'instrument']:
        allowed &= ieeg[column].to_numpy()[:, None] == meg[column].to_numpy()[None, :]
    if max_age_gap is not None:
        allowed &= gaps <= max_age_gap
    try:
        rows, cols = linear_sum_assignment(np.where(allowed, gaps, np.inf))
    except ValueError as error:
        raise ValueError('No complete assignment satisfies exact matching and the age caliper. Review overlap and constraints.') from error
    if len(rows) != len(ieeg) or not allowed[rows, cols].all():
        raise RuntimeError('Solver did not return a complete valid assignment.')
    pairs = pd.concat([ieeg.iloc[rows].reset_index(drop=True).add_prefix('ieeg_'),
                       meg.iloc[cols].reset_index(drop=True).add_prefix('meg_')], axis=1)
    pairs['age_gap_years'] = gaps[rows, cols]
    pairs['meg_minus_ieeg_age'] = pairs['meg_age'] - pairs['ieeg_age']
    assert pairs['ieeg_subject'].is_unique and pairs['meg_subject'].is_unique
    return pairs, counts

matched_pairs, stratum_counts = optimal_demographic_match(ieeg_eligible, meg_eligible, MAX_AGE_GAP)
matched_meg_subjects = matched_pairs['meg_subject'].tolist()
meg_matched = meg_eligible.set_index('subject').loc[matched_meg_subjects].reset_index()
print('Available participants by exact-match stratum:'); display(stratum_counts)
print(f'Matched {len(matched_pairs)} distinct pairs')
display(matched_pairs)
display(matched_pairs[['age_gap_years', 'meg_minus_ieeg_age']].describe())


## Balance and pair quality

Signed standardized mean differences (SMDs) compare MEG minus iEEG. The denominator is the pooled **unmatched** standard deviation and is held fixed after matching. Binary covariates use Bernoulli variances; multicategory sex is evaluated with one indicator per category. An absolute SMD of 0.1 is a descriptive reference, not a guarantee of adequate balance. Review raw age distributions and the largest pairwise gaps as well. Exact categorical matching guarantees identical categorical proportions among retained pairs; it does not guarantee age balance or equivalence of questionnaire constructs.


In [ ]:
def balance_table(ieeg, meg, selected):
    features = {'Age (years)': (ieeg['age'], meg['age'], selected['age'])}
    for column in ['sex', 'instrument']:
        for level in sorted(set(ieeg[column]) | set(meg[column])):
            features[f'{column}: {level}'] = tuple((frame[column] == level).astype(float)
                                                    for frame in (ieeg, meg, selected))
    records = []
    for name, (target, before, after) in features.items():
        binary = name != 'Age (years)'
        var = lambda x: x.mean() * (1 - x.mean()) if binary else x.var(ddof=1)
        scale = np.sqrt((var(target) + var(before)) / 2)
        def smd(value):
            delta = value.mean() - target.mean()
            return delta / scale if scale > 0 else (0.0 if delta == 0 else np.nan)
        records.append({'covariate': name, 'iEEG_mean_or_proportion': target.mean(),
                        'MEG_before': before.mean(), 'MEG_matched': after.mean(),
                        'SMD_before': smd(before), 'SMD_matched': smd(after)})
    return pd.DataFrame(records).set_index('covariate')

balance = balance_table(ieeg_eligible, meg_eligible, meg_matched)
display(balance.round(3))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
y = np.arange(len(balance))
for col, label in [('SMD_before', 'Before'), ('SMD_matched', 'Matched')]:
    axes[0].scatter(balance[col].abs(), y, label=label)
axes[0].axvline(.1, color='gray', linestyle=':')
axes[0].set(yticks=y, yticklabels=balance.index, xlabel='Absolute SMD (fixed denominator)')
axes[0].legend()
for frame, label in [(ieeg_eligible, 'iEEG'), (meg_eligible, 'MEG before'), (meg_matched, 'MEG matched')]:
    ages = np.sort(frame['age'].to_numpy())
    axes[1].step(ages, np.arange(1, len(ages)+1)/len(ages), where='post', label=label)
axes[1].set(xlabel='Age (years)', ylabel='Empirical cumulative proportion')
axes[1].legend()
axes[2].scatter(matched_pairs['ieeg_age'], matched_pairs['meg_age'])
limits = [min(ieeg_eligible.age.min(), meg_matched.age.min()),
          max(ieeg_eligible.age.max(), meg_matched.age.max())]
axes[2].plot(limits, limits, 'k:', linewidth=1)
axes[2].set(xlabel='iEEG age (years)', ylabel='Matched MEG age (years)')
fig.tight_layout()
plt.show()
print('Largest age gaps:')
display(matched_pairs.sort_values('age_gap_years', ascending=False).head(10))


In [ ]:
# Optional reproducible outputs; matching code remains entirely in this notebook.
if EXPORT_RESULTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    matched_pairs.to_csv(OUTPUT_DIR / 'matched_pairs.csv', index=False)
    balance.to_csv(OUTPUT_DIR / 'balance.csv')
    meg_matched.to_csv(OUTPUT_DIR / 'selected_meg_demographics.csv', index=False)
    pd.Series(matched_meg_subjects, name='subject').to_csv(OUTPUT_DIR / 'selected_meg_subjects.csv', index=False)
    print(f'Saved results to {OUTPUT_DIR.resolve()}')
